1. Write a function that:
    - Takes a block number as input
    - Returns the timestamp of when it was mined (human-readable)
    - Hint: We did this in the ERC20 transfer event analysis section

In [ ]:
from web3 import Web3
import pandas as pd
import requests
from dotenv import load_dotenv
import os
from datetime import datetime

# Loading environment variables from .env file

load_dotenv()
GATEWAY = os.getenv("GATEWAY_URL")

print(f'GATEWAY imported as {GATEWAY}')

In [2]:
# Connecting to the blockchain

w3 = Web3(Web3.HTTPProvider(GATEWAY))
print(f'Is the connection successful?: {w3.is_connected()}') # Should return True

Is the connection successful?: True


In [3]:
def turn_to_checksum(address: str) -> str:
    """Convert an Ethereum address to its checksum format"""
    if not Web3.is_checksum_address(address): # check if already checksum
        return Web3.to_checksum_address(address) # if not we return checksum address
    return address

ACCOUNT_ADDRESS_RAW = "0xb13ae9be79BD2a27d9fA258AFE0b4BC6f7380441"
ACCOUNT_ADDRESS = turn_to_checksum(ACCOUNT_ADDRESS_RAW)

In [4]:
# We can get the timestamp by looking up the block

def get_block_timestamp(block_number: int) -> str:

    # Fetch the block by number
    block = w3.eth.get_block(block_number) 

    # Extract Unix timestamp
    timestamp = block.timestamp

    # Convert to human-readable string (UTC)
    return datetime.utcfromtimestamp(timestamp).strftime("%Y-%m-%d %H:%M:%S UTC")

In [5]:
print(get_block_timestamp(23188524))

2025-08-21 09:21:35 UTC


C:\Users\FFFO CASHIER PT\AppData\Local\Temp\ipykernel_7888\3671900980.py:12: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  return datetime.utcfromtimestamp(timestamp).strftime("%Y-%m-%d %H:%M:%S UTC")


2. Calculate average gas price of last 10 blocks

In [6]:
def average_gas_price(latest_block_N: int = 10):
    # Get the latest block number
    latest_block = w3.eth.block_number
    gas_prices = []
    
    for i in range(latest_block, latest_block - latest_block_N, -1):
        block = w3.eth.get_block(i, full_transactions=True)
        txs = block['transactions']
        if len(txs) > 0:
            gas_prices.extend([tx['gasPrice'] for tx in txs])

    if gas_prices:
        return sum(gas_prices) // len(gas_prices)
    return 0

In [7]:
print('Average gas price:', average_gas_price(), "wei")

Average gas price: 3112099672 wei


In [8]:
# Get average gas price (Wei)
avg_gas_price_wei = average_gas_price()

# Convert to Ether
avg_gas_price_eth = w3.from_wei(avg_gas_price_wei, 'ether')

print('Average gas price:', avg_gas_price_eth, 'ETH')


Average gas price: 3.257872116E-9 ETH


3. Create a simple whale detector function
    - The function should:
        - Check if a wallet has more than 100 ETH
        - Check if a wallet has more than 1M USDC
        - If either is True, the wallet should be tagged a "Whale" (can simply print "Whale")
        - Optionally, can add more categories (goldfish, dolphin, small whale, large whale, etc.)
    - Test the function with your address or addresses you find on [etherscan.io](https://etherscan.io/token/0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48#balances)
    - Bonus:
        - Create a function that takes in several addresses, checks its category, and appends the address and tag to a dictionary (Address is key, tag is the value)

In [17]:
# Token Balance

WETH_CONTRACT_ADDRESS_RAW = '0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2'
WETH_CONTRACT_ADDRESS = turn_to_checksum(WETH_CONTRACT_ADDRESS_RAW)

USDC_ADDRESS = "0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48" # Ethereum USDC Address
ERC20_ABI = [
    {
        "constant": True,
        "inputs": [{"name": "_owner", "type": "address"}],
        "name": "balanceOf",
        "outputs": [{"name": "balance", "type": "uint256"}],
        "payable": False,
        "stateMutability": "view",
        "type": "function"
    },
    {
        "constant": True,
        "inputs": [],
        "name": "totalSupply",
        "outputs": [
            {
                "name": "",
                "type": "uint256"
            }
        ],
        "payable": False,
        "stateMutability": "view",
        "type": "function"
    },
    {
        "constant": True,
        "inputs": [],
        "name": "decimals",
        "outputs": [
            {
                "name": "",
                "type": "uint8"
            }
        ],
        "payable": False,
        "stateMutability": "view",
        "type": "function"
    },
]
usdc_contract = w3.eth.contract(address=USDC_ADDRESS, abi=ERC20_ABI) # Create contract client to interact with the contract functions

# Get USDC balance
decimals = usdc_contract.functions.decimals().call()
raw_balance = usdc_contract.functions.balanceOf(WETH_CONTRACT_ADDRESS).call() # pass address to balanceOf function
normalized_balance = raw_balance / 10 ** decimals
# ETH balance
eth_balance = w3.eth.get_balance(WETH_CONTRACT_ADDRESS) / 1e18

print(f'Balance for {WETH_CONTRACT_ADDRESS} → ETH: {eth_balance:.2f}, USDC: {normalized_balance:,.0f}')

Balance for 0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2 → ETH: 2222031.64, USDC: 6,156


In [29]:
def whale_detector(address):
    # Get ETH balance
    eth_balance = w3.eth.get_balance(address) / 1e18

    # Get USDC balance
    decimals = usdc_contract.functions.decimals().call()
    raw_balance = usdc_contract.functions.balanceOf(address).call()
    usdc_balance = raw_balance / 10 ** decimals

    # Check conditions
    if eth_balance > 1000 or usdc_balance > 1_000_000:
        return 'Mega Whale'
    elif eth_balance > 1000 or usdc_balance > 100_000:
        return 'Whale'
    elif eth_balance > 100 or usdc_balance > 10_000:
        return 'Dolphin'
    else:
        return 'Goldfish'

In [30]:
print(whale_detector("0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2"))

Mega Whale


In [31]:
def classify_addresses(addresses):
    results = {}
    for addr in addresses:
        results[addr] = whale_detector(addr)
    return results

In [35]:
print(classify_addresses(['0x7a250d5630B4cF539739dF2C5dAcb4c659F2488D']))


{'0x7a250d5630B4cF539739dF2C5dAcb4c659F2488D': 'Goldfish'}
